<a href="https://colab.research.google.com/github/NathanCarvalhoo/Inteligencia-Artificial/blob/main/Fishmorph_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importando Dataset do Kaggle**



In [ ]:
import kagglehub

path = kagglehub.dataset_download("menegidio/fishmorph-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fishmorph-dataset' dataset.
Path to dataset files: /kaggle/input/fishmorph-dataset


# Importando Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Carregando dados

In [ ]:
import pandas as pd

df = pd.read_csv('FISHMORPH_Family_Dataset.csv')

df.head()

,MBl,BEl,VEp,REs,OGp,RMl,BLs,PFv,PFs,CPt,Family
0,130.0,4.579113,0.622642,0.375019,0.622642,0.868659,0.484981,0.273585,0.144108,3.348991,Cyprinidae
1,11.5,2.796068,0.650700,0.353780,0.509800,0.411157,0.722461,0.330169,0.221095,2.525592,Cichlidae
2,6.6,4.564329,0.518187,0.312002,0.203985,0.413221,0.591495,0.105517,0.188193,2.104580,Cyprinidae
3,10.9,4.390422,0.548247,0.244804,0.168959,0.324728,0.642991,0.097450,0.193049,1.787062,Cyprinidae
4,18.9,4.410271,0.542353,0.292058,0.183790,0.398567,0.685672,0.087643,0.170648,2.968906,Cyprinidae


**Tipos de dados**

*   MBl -	Comprimento da mandíbula
*   BEl	- Comprimento do olho
*   VEp	- Posição de abertura da barbatana ventral
*   REs	- Comprimento do espinho da barbatana dorsal
*   OGp	- Vaga de emprego em Gill
*   RMl	- Comprimento da barbatana peitoral
*   BLs	- Formato do corpo (relação comprimento/largura)
*   PFv	- Posição da barbatana peitoral
*   PFs	- Formato da barbatana peitoral
*   CPt	- Tipo de pedúnculo caudal
*   Família	- Classificação taxonômica ao nível da família


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8342 entries, 0 to 8341
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   MBl     8342 non-null   float64
 1   BEl     8342 non-null   float64
 2   VEp     8342 non-null   float64
 3   REs     8342 non-null   float64
 4   OGp     8342 non-null   float64
 5   RMl     8342 non-null   float64
 6   BLs     8342 non-null   float64
 7   PFv     8342 non-null   float64
 8   PFs     8342 non-null   float64
 9   CPt     8342 non-null   float64
 10  Family  8342 non-null   object 
dtypes: float64(10), object(1)
memory usage: 717.0+ KB


# Valores Ausentes

Identificando possíveis valores nulos

In [ ]:
df.isnull().sum()


,0
MBl,0
BEl,0
VEp,0
REs,0
OGp,0
RMl,0
BLs,0
PFv,0
PFs,0
CPt,0


# Preparando dados

In [ ]:
X = df.drop(columns=["Family"])
y = df["Family"]

print("Características:", X.shape)
print("Famílias:", y.nunique())

print(y.value_counts().head(10))

Características: (8342, 10)
Famílias: 197
Family
Cyprinidae        1545
Cichlidae         1060
Characidae         732
Loricariidae       427
Percidae           190
Gobiidae           187
Callichthyidae     159
Rivulidae          158
Poeciliidae        154
Nemacheilidae      145
Name: count, dtype: int64


**Tratando dados**

In [ ]:
contagem = y.value_counts()

familias_validas = contagem[contagem >= 10].index

df_modelo = df[df["Family"].isin(familias_validas)].copy()

X = df_modelo.drop(columns=["Family"])
y = df_modelo["Family"]

print("Registros utilizados:", len(df_modelo))
print("Número de famílias:", y.nunique())

Registros utilizados: 7999
Número de famílias: 90


# Treinamento e testes

80% treinamento, 20% testes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", X_train.shape)
print("Teste:", X_test.shape)

Treinamento: (6399, 10)
Teste: (1600, 10)


# GridSearchCV

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

parametros = {
    "svm__kernel": ["rbf", "linear"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=parametros,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Melhores parâmetros:")
print(grid.best_params_)

print("Melhor acurácia média na validação cruzada:")
print(grid.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Melhores parâmetros:
{'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Melhor acurácia média na validação cruzada:
0.6729178068803753


# Acurácia

In [ ]:
y_pred = grid.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)

print(f"Acurácia no teste (%): {acuracia * 100:.2f}%")

Acurácia no teste (%): 68.62%
